# Hierarchical Difference-in-Differences

This notebook demonstrates the `HierarchicalDifferenceInDifferences` estimator in CausalPy, which handles Difference-in-Differences settings where treatment is assigned at a group level and outcomes are observed for lower-level units nested within those groups.

Our example is a store-level marketing campaign where customers are nested within stores. The central question is: **what was the average effect of the campaign, while accounting for the fact that customers in the same store tend to move together?**

In this notebook, the non-hierarchical DiD is fitted directly to customer-month outcomes without representing the store-level structure at which treatment is assigned. The fixed-effects DiD controls for persistent store differences, while the hierarchical model explicitly represents store-level variation and partially pools information across stores. All three models retain the DiD estimand, but represent the store structure differently.


In [ ]:
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import causalpy as cp
from causalpy.data.simulate_data import generate_hdid_data

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'
seed = 20260296

## The Hierarchical DiD Problem

A standard Difference-in-Differences design compares treated and control units before and after an intervention. In this example, the intervention is a marketing campaign assigned to stores.

The hierarchy matters because treatment is assigned to stores, while outcomes are measured for customers within stores. Customers in the same store share the same local market, staff, merchandising, foot traffic, and campaign implementation. In the observation-level DiD used as a baseline below, each customer-month outcome enters the model directly without an explicit store-level component. Because treatment varies at the store level, that specification does not represent this shared store context and may understate uncertainty.

A store fixed-effects model addresses part of the problem by allowing each store to have its own baseline. However, fixed effects do not pool information across stores, and they do not directly represent variation in treatment response as a modeled quantity. A hierarchical DiD model estimates the population-average treatment effect while allowing store-level baselines and treatment responses to vary around that average.

In this notebook, we first fit the hierarchical estimator, then compare it with non-hierarchical DiD baselines on the same analysis panel.


## Identifying Assumptions

The hierarchical model changes the outcome model, not the identifying design. We still need a credible DiD setup: treated and control stores should be comparable enough that, without the campaign, their outcomes would have followed parallel trends.

Key assumptions in this example:

- **Parallel trends**: Treated and control stores would have followed similar changes over time without the campaign.
- **Group-level treatment**: Treatment is assigned at the store level. Customers within a treated store receive the same campaign status in a given period.
- **Simultaneous adoption**: All treated stores begin treatment in the same post-treatment period.
- **Stable membership**: Customers do not switch stores during the analysis window.
- **Balanced panel**: Every selected customer is observed in every selected month.
- **Partial-pooling structure**: Store-level baseline and treatment-response deviations are reasonably represented as draws from a common distribution.

:::{important}
The random effects do not make a weak DiD design causal. They help model clustered outcomes and heterogeneous responses once the DiD design assumptions are plausible.
:::


## Generate Synthetic Hierarchical Panel Data

We'll create a compact synthetic panel with:

- 10 stores observed over 6 months
- 5 customers per store
- 5 treated stores and 5 control stores
- 3 pre-treatment months and 3 post-treatment months
- A known average treatment effect of 4 purchase-amount units
- Store-level treatment-response deviations with a standard deviation of 1.5 units

The panel is intentionally small so the full Bayesian workflow remains runnable in the documentation while still preserving the nested design structure.


In [ ]:
n_stores_total = 10
n_stores_treated = 5
n_months = 6
pre_months = 3
treatment_start_month = pre_months + 1
customers_per_store = 5
true_att = 4.0
treatment_response_sd = 1.5
outcome_noise_sd = 2.0

panel, simulation = generate_hdid_data(
    seed=seed,
    n_stores_total=n_stores_total,
    n_stores_treated=n_stores_treated,
    n_months=n_months,
    pre_months=pre_months,
    units_min=customers_per_store,
    units_max=customers_per_store,
    units_lam=float(customers_per_store),
    att_effect=true_att,
    sigma_treatment_slope=treatment_response_sd,
    sigma_noise=outcome_noise_sd,
)

panel.head()

Treatment is assigned at the store level, while each row records a customer outcome for a given month. The fitted hierarchical model uses store-level random effects to represent variation shared by observations from the same store.


In [ ]:
def add_model_columns(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.assign(
        customer_age_centered=lambda data: (
            data["customer_age"] - data["customer_age"].mean()
        ),
        customer_tenure_centered=lambda data: (
            data["customer_tenure"] - data["customer_tenure"].mean()
        ),
        pre_purchase_history_centered=lambda data: (
            data["pre_purchase_history"] - data["pre_purchase_history"].mean()
        ),
        log_store_size=lambda data: np.log(data["store_size"]),
        log_store_size_centered=lambda data: (
            np.log(data["store_size"]) - np.log(data["store_size"]).mean()
        ),
    )


analysis_panel = (
    panel.pipe(add_model_columns)
    .sort_values(["store_id", "customer_id", "month"])
    .assign(
        unit=lambda data: data["customer_id"],
        did=lambda data: data["post_treatment"] * data["treated"],
    )
    .reset_index(drop=True)
)

analysis_panel.head()

The generated data are already the analysis panel. Customers are nested in stores, treated and control stores are balanced, and the panel includes both pre- and post-treatment periods. The treatment effect is identified by the interaction between treatment group and post-treatment period.


## Visualize Clustering and Treatment Pattern

Before fitting a model, it is useful to inspect two design features: when treatment turns on, and whether stores differ meaningfully in their baseline outcome paths.

The first plot is a treatment-status heatmap. A clean simultaneous-adoption DiD design should show treated stores switching on at the same calendar time, while control stores remain untreated throughout. The vertical dashed line marks the intervention boundary.


In [ ]:
store_order = (
    analysis_panel[["store_id", "treated"]]
    .drop_duplicates()
    .sort_values(["treated", "store_id"])["store_id"]
)

treatment_grid = (
    analysis_panel[["store_id", "treated", "month", "post_treatment"]]
    .drop_duplicates()
    .assign(active_treatment=lambda data: data["treated"] * data["post_treatment"])
    .pivot(index="store_id", columns="month", values="active_treatment")
    .loc[store_order]
)

month_starts = treatment_grid.columns.to_numpy()

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(
    treatment_grid.values,
    aspect="auto",
    cmap="Greys",
    interpolation="nearest",
    vmin=0,
    vmax=1,
    extent=(
        month_starts[0],
        month_starts[-1] + 1,
        len(treatment_grid.index) - 0.5,
        -0.5,
    ),
)
ax.axvline(treatment_start_month, color="black", linestyle="--", linewidth=1)
ax.set(title="Treatment pattern by store", xlabel="Month", ylabel="Store")
ax.set_xticks(month_starts, labels=treatment_grid.columns)
ax.set_yticks(np.arange(len(treatment_grid.index)), labels=treatment_grid.index)

colorbar = fig.colorbar(im, ax=ax, ticks=[0, 1], pad=0.02)
colorbar.ax.set_yticklabels(["Untreated", "Treated"])
colorbar.set_label("Treatment status")

fig.tight_layout()
plt.show()

The next plot shows store-level mean purchase trajectories. Thin lines show individual stores and thick lines show treated/control averages.

In a real analysis, this plot is a first diagnostic for the parallel trends story. We would like the treated and control averages to move similarly before the intervention. After the intervention, treated stores should separate from controls if the campaign has an effect. Store-to-store variation around the average paths is exactly the structure the hierarchical model is designed to represent.

In [ ]:
store_time = (
    analysis_panel.groupby(["store_id", "treated", "month"], observed=True)[
        "purchase_amount"
    ]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 4))
for treated, color, label in [
    (0, "C0", "Control stores"),
    (1, "C3", "Treated stores"),
]:
    subset = store_time.loc[store_time["treated"] == treated]
    for _, store_frame in subset.groupby("store_id", observed=True):
        ax.plot(
            store_frame["month"],
            store_frame["purchase_amount"],
            color=color,
            alpha=0.22,
            linewidth=1,
        )
    mean_line = subset.groupby("month", observed=True)["purchase_amount"].mean()
    ax.plot(mean_line.index, mean_line.values, color=color, linewidth=3, label=label)

ax.axvline(treatment_start_month, color="black", linestyle="--", linewidth=1)
ax.set(
    title="Store-level purchase trajectories",
    xlabel="Month",
    ylabel="Mean purchase amount",
)
ax.legend();

## Fit the Hierarchical DiD Model

We'll use a hierarchical regression model with store-level random effects for the clustered DiD design.

The formula specifies the outcome model. In hierarchical DiD, the key fixed effect is the DiD interaction, `post_treatment:treated`. This coefficient is the population-average {term}`ATT`.

The formula also includes one random-effects term, `(1 + post_treatment:treated | store_id)`. This says that stores may differ in their baseline outcome level and may also differ in their treatment response around the population-average effect.

The experiment arguments identify the panel structure used by the estimator: the time column, the unit column, the treatment indicator, and the post-treatment indicator. The random-effects grouping variable, `store_id`, is read from the formula.

:::{note}
The `random_seed` keyword argument for the PyMC sampler is not necessary. We use it here so that the results are reproducible. The sampling settings below are intentionally small so the notebook remains fast. The posterior summaries and diagnostics should be read as examples of the workflow, not as final inferential evidence. For real analysis, increase `draws` and `tune` and inspect the diagnostics carefully.
:::


In [ ]:
from causalpy.formula import parse_formula

hdid_formula = (
    "purchase_amount ~ 1 + post_treatment + treated + post_treatment:treated "
    "+ (1 + post_treatment:treated | store_id)"
)

matrices = parse_formula(hdid_formula, analysis_panel)

formula_summary = pd.DataFrame(
    {
        "model component": [
            "outcome",
            "fixed ATT term",
            "random-effects group",
            "random intercept",
            "random treatment-response deviation",
            "number of groups",
        ],
        "formula term": [
            matrices.metadata["outcome_name"],
            "post_treatment:treated",
            matrices.metadata["group"]["variable"],
            "1|store_id",
            "post_treatment:treated|store_id",
            matrices.metadata["group"]["n_groups"],
        ],
    }
)
formula_summary

The table above checks the pieces of the model specification:

- `purchase_amount` is the observed outcome;
- `post_treatment:treated` is included as a fixed effect, so the model estimates a population-average ATT;
- `store_id` is the grouping variable for the random effects;
- `post_treatment:treated|store_id` is included as a random slope, so store-level treatment responses can vary around the average effect.

The store-level random slope should not be read as a separate standalone ATT for each store. It is a partially pooled deviation from the population-average ATT.


In [ ]:
sample_kwargs = {
    "draws": 50,
    "tune": 50,
    "chains": 2,
    "cores": 1,
    "target_accept": 0.9,
    "progressbar": False,
    "random_seed": seed,
}

hdid_model = cp.pymc_models.HierarchicalLinearRegression(sample_kwargs=sample_kwargs)

result = cp.HierarchicalDifferenceInDifferences(
    data=analysis_panel,
    formula=hdid_formula,
    time_variable_name="month",
    unit_variable_name="customer_id",
    model=hdid_model,
)

After fitting the hierarchical model, the result contains the population-average DiD effect, store-level random effects, residual scale, random-effect scales, and an ICC estimate.

## Plot the Result

The default plot gives a first visual summary of the fitted experiment. Use it before inspecting coefficient tables or group-level diagnostics.


In [ ]:
fig, ax = result.plot()

## View Summary Statistics

The summary reports the main causal effect and the model quantities used to interpret the hierarchical fit. The causal impact is the posterior distribution of the fixed `post_treatment:treated` coefficient.


In [ ]:
result.summary(round_to=2)

## Effect Summary

Get a prose summary of the causal effect. For hierarchical DiD, this summary describes the population-average post-treatment effect for treated groups.

Since this is synthetic data, we can also compare the estimate to the known data-generating effect, `true_att`.


In [ ]:
att_summary = az.summary(result.causal_impact, hdi_prob=0.94)
att_summary

In [ ]:
effect = result.effect_summary()
print(effect.text)
effect.table.round(2)

The prose effect summary is useful for contexts where a coefficient table is too low level. It reports the estimated average treatment effect, uncertainty interval, and posterior probability that the effect is positive.

For hierarchical DiD, this summary is about the **population-average campaign effect**. Store-specific departures from this average are interpreted separately through the group effects.


## Comparison to Non-Hierarchical DiD

We compare three PyMC estimators on the same compact analysis panel:

1. **Observation-level DiD** with `cp.DifferenceInDifferences`, fitted directly to customer-month outcomes without store-level effects.
2. **Store fixed-effects DiD** with `cp.PanelRegression`, using store and month fixed effects.
3. **Hierarchical DiD** with `cp.HierarchicalDifferenceInDifferences`, which models store-level random effects and partial pooling.

These estimators answer closely related questions under different modeling assumptions.


In [ ]:
baseline_sample_kwargs = {
    "draws": 1000,
    "tune": 2000,
    "chains": 4,
    "cores": 4,
    "target_accept": 0.9,
    "progressbar": False,
    "random_seed": seed,
}

naive_did_result = cp.DifferenceInDifferences(
    data=analysis_panel.copy(),
    formula="purchase_amount ~ 1 + treated*post_treatment",
    time_variable_name="month",
    group_variable_name="treated",
    model=cp.pymc_models.LinearRegression(sample_kwargs=baseline_sample_kwargs),
)

store_fe_result = cp.PanelRegression(
    data=analysis_panel.copy(),
    formula="purchase_amount ~ did + customer_age_centered + customer_tenure_centered + pre_purchase_history_centered",
    unit_fe_variable="store_id",
    time_fe_variable="month",
    fe_method="demeaned",
    model=cp.pymc_models.LinearRegression(sample_kwargs=baseline_sample_kwargs),
)


def posterior_coefficient_summary(
    idata: az.InferenceData, coefficient: str
) -> pd.Series:
    summary = az.summary(
        idata.posterior["beta"].sel(coeffs=coefficient),
        hdi_prob=0.94,
    )
    return summary.iloc[0]


naive_summary = az.summary(naive_did_result.causal_impact, hdi_prob=0.94).iloc[0]
store_fe_summary = posterior_coefficient_summary(store_fe_result.model.idata, "did")
hdid_summary = az.summary(result.causal_impact, hdi_prob=0.94).iloc[0]

estimator_comparison = pd.DataFrame(
    [
        {
            "approach": "Hierarchical DiD (PyMC)",
            "att_estimate": hdid_summary["mean"],
            "posterior_sd": hdid_summary["sd"],
            "hdi_3%": hdid_summary["hdi_3%"],
            "hdi_97%": hdid_summary["hdi_97%"],
        },
        {
            "approach": "Naive DiD (PyMC)",
            "att_estimate": naive_summary["mean"],
            "posterior_sd": naive_summary["sd"],
            "hdi_3%": naive_summary["hdi_3%"],
            "hdi_97%": naive_summary["hdi_97%"],
        },
        {
            "approach": "Store fixed effects DiD (PyMC)",
            "att_estimate": store_fe_summary["mean"],
            "posterior_sd": store_fe_summary["sd"],
            "hdi_3%": store_fe_summary["hdi_3%"],
            "hdi_97%": store_fe_summary["hdi_97%"],
        },
    ]
)
estimator_comparison

The comparison table reports posterior summaries for the DiD coefficient from each PyMC model.

The point estimates should generally be in the same neighborhood because the synthetic data were generated from a clean simultaneous-adoption design. Their uncertainty can nevertheless differ. In this comparison, the observation-level DiD fits every customer-month outcome without store-level random effects. The fixed-effects model controls for persistent store differences but does not pool store-level treatment-response deviations. The hierarchical model estimates the population effect and store-level heterogeneity jointly.

## Group Effects and Variance Components

The forest plot shows store-level deviations for the treatment-response random slope. These are not separate store-level treatment effects estimated in isolation. They are partially pooled deviations from the population-average ATT.

All stores remain visible in the plot. For treated stores, the treatment-response deviation is informed by their observed post-treatment outcomes. For control stores, the treatment interaction is always zero, so the displayed deviation is a prior-centered, model-implied quantity rather than a separately identified treatment response.

Partial pooling is important for business interpretation. Stores with little information are pulled more strongly toward the overall average, while stores with more evidence can deviate more. This reduces the tendency to overinterpret noisy top/bottom rankings.

The variance component plot shows posterior uncertainty for the store-level scales and residual scale. The ICC summarizes the share of residual variation attributable to store-level clustering.

In [ ]:
fig, ax = result.plot_group_effects(random_coeff="post_treatment:treated|store_id")
ax.set_title("Store-level treatment-response deviations");

In [ ]:
fig, ax = result.plot_variance_components()
fig.suptitle("Posterior variance components", y=1.02);

In [ ]:
icc_posterior = result.icc
icc_summary = az.summary(icc_posterior, hdi_prob=0.94)
icc_mean = float(icc_summary["mean"].iloc[0])

pd.DataFrame(
    {
        "quantity": ["Realized sample ICC", "ICC posterior mean", "interpretation"],
        "value": [
            simulation["empirical_icc"],
            icc_mean,
            f"About {icc_mean:.0%} of residual variation is attributed to store-level clustering in this fitted model.",
        ],
    }
)

The ICC interpretation should be read alongside the variance components. A larger ICC indicates that store-level clustering remains important after accounting for the fixed effects in the model. In practical terms, this means two customers from the same store are expected to be more similar than two customers from different stores, even after adjusting for observed covariates and treatment timing.

This is one of the main reasons to use a hierarchical DiD model in this analysis: outcomes are observed at the customer-month level, while treatment is assigned and important outcome variation is shared at the store level.

## Diagnostics

Before interpreting the model, inspect convergence diagnostics. `r_hat` values should be close to 1 and effective sample sizes should be comfortably large. Divergences, low ESS, or high `r_hat` are signs that the sampler needs more tuning, stronger priors, better scaling, or a revised parameterization.

In [ ]:
diagnostics = az.summary(
    result.idata,
    var_names=["beta_fixed", "sigma_random", "sigma_fixed"],
    filter_vars="like",
)

diagnostics[["mean", "sd", "ess_bulk", "r_hat"]].head(12)

## Interpreting Store-Level Deviations

The hierarchical model separates the population-average campaign effect from store-specific deviations. This is useful for inspecting stores with likely over- or under-performance while retaining partial pooling.

In a campaign setting, this table can guide follow-up questions:

- Are high-performing stores concentrated in a region or store format?
- Are low-performing stores under-implementing the campaign?
- Do stores with high baseline purchase levels respond differently?
- Is heterogeneity large enough to justify targeted operational changes?

These rankings are descriptive posterior summaries, not separate store-level causal experiments. Store-level deviations are posterior quantities with uncertainty.


In [ ]:
random_att = result.group_effects.sel(
    random_coeffs="post_treatment:treated|store_id"
).squeeze("treated_units", drop=True)
random_att_hdi = az.hdi(random_att)["beta_random"]
group_effects = pd.DataFrame(
    {
        "group": random_att.coords["groups"].to_numpy(),
        "parameter": random_att.coords["random_coeffs"].item(),
        "mean": random_att.mean(dim=("chain", "draw")).to_numpy(),
        "hdi_3%": random_att_hdi.sel(hdi="lower").to_numpy(),
        "hdi_97%": random_att_hdi.sel(hdi="higher").to_numpy(),
    }
)

top_bottom = pd.concat(
    [
        group_effects.nsmallest(5, "mean").assign(
            rank_group="Lowest treatment-response deviations"
        ),
        group_effects.nlargest(5, "mean").assign(
            rank_group="Highest treatment-response deviations"
        ),
    ]
)

top_bottom[["rank_group", "group", "parameter", "mean", "hdi_3%", "hdi_97%"]]

The top and bottom rows show stores with the largest posterior deviations from the average treatment response. A store in the high-deviation group is estimated to have responded more strongly than the average treated store, while a low-deviation store responded less strongly.

Because the estimates are partially pooled, the ranking is less volatile than a fully unpooled store-by-store analysis. These rankings are descriptive posterior summaries, not separate store-level causal experiments. The HDI columns should still be used when deciding whether differences are meaningful enough to act on.


## Key Takeaways

1. **Hierarchical DiD is for clustered DiD designs** where treatment is assigned at a grouped level and outcomes are observed below that level.

2. **The formula and experiment arguments play different roles**. The formula specifies fixed and random effects; the experiment arguments identify the panel structure used for validation and interpretation.

3. **Naive and fixed-effects DiD are useful baselines**, but they do not provide the same partial-pooling interpretation or variance-component diagnostics as the hierarchical model.

4. **Group effects are deviations from the population effect**, not independent store-by-store experiments. This makes top/bottom performer analysis less noisy than fully unpooled rankings.

5. **Validation is part of the estimator**. Simultaneous-adoption HDiD should reject staggered adoption rather than silently fit the wrong design.

## References

- Bertrand, M., Duflo, E., and Mullainathan, S. (2004). How much should we trust differences-in-differences estimates?
- Gelman, A., and Hill, J. (2006). Data Analysis Using Regression and Multilevel/Hierarchical Models.
- PyMC Difference-in-Differences example: https://www.pymc.io/projects/examples/en/latest/causal_inference/difference_in_differences.html
- Formulaic documentation: https://matthewwardrop.github.io/formulaic/